In [1]:
!git clone https://github.com/FlagOpen/FlagEmbedding.git
%cd FlagEmbedding
!pip install -e .

Cloning into 'FlagEmbedding'...
remote: Enumerating objects: 11544, done.
remote: Counting objects: 100% (3437/3437), done.
remote: Compressing objects: 100% (1273/1273), done.
remote: Total 11544 (delta 2220), reused 2164 (delta 2164), pack-reused 8107 (from 1)
Receiving objects: 100% (11544/11544), 51.36 MiB | 35.04 MiB/s, done.
Resolving deltas: 100% (6594/6594), done.
/kaggle/working/FlagEmbedding
Obtaining file:///kaggle/working/FlagEmbedding
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.9 MB/s eta 0:00:00
  Running setup.py develop for FlagEmbedding


In [16]:
!python -m FlagEmbedding.finetune.embedder.encoder_only.base --help | grep save

                   [--save_strategy {no,steps,epoch,best}]
                   [--save_steps SAVE_STEPS]
                   [--save_total_limit SAVE_TOTAL_LIMIT]
                   [--save_on_each_node [SAVE_ON_EACH_NODE]]
                   [--save_only_model [SAVE_ONLY_MODEL]]
                   [--hub_strategy {end,every_save,checkpoint,all_checkpoints}]
  --save_strategy {no,steps,epoch,best}, --save-strategy {no,steps,epoch,best}
                        The checkpoint save strategy to use. (default: steps)
  --save_steps SAVE_STEPS, --save-steps SAVE_STEPS
  --save_total_limit SAVE_TOTAL_LIMIT, --save-total-limit SAVE_TOTAL_LIMIT
                        `save_total_limit=5` and
                        model. When `save_total_limit=1` and
                        checkpoints are saved: the last one and the best one
  --save_on_each_node [SAVE_ON_EACH_NODE], --save-on-each-node [SAVE_ON_EACH_NODE]
                        save models and checkpoints on each node, or only on
  --save_on

In [17]:
!torchrun --nproc_per_node=1 \
-m FlagEmbedding.finetune.embedder.encoder_only.base \
--model_name_or_path BAAI/bge-base-en-v1.5 \
--cache_dir /root/.cache/huggingface/hub \
--train_data /kaggle/input/datasets/madhavkumar244/madhavscuadhn/train_bge_hn.jsonl \
--cache_path /kaggle/working/cache \
--train_group_size 8 \
--query_max_len 384 \
--passage_max_len 384 \
--pad_to_multiple_of 8 \
--knowledge_distillation False \
--output_dir /kaggle/working/bge_legal \
--learning_rate 1e-5 \
--fp16 \
--num_train_epochs 2 \
--per_device_train_batch_size 8 \
--dataloader_drop_last True \
--warmup_ratio 0.1 \
--gradient_checkpointing \
--logging_steps 20 \
--save_strategy no \
--temperature 0.02 \
--sentence_pooling_method cls \
--normalize_embeddings True

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
08/06/2026 16:13:43 - WARNING - FlagEmbedding.abc.finetune.embedder.AbsRunner -   Process rank: -1, device: cuda:0, n_gpu: 1, distributed training: False, 16-bits training: True
08/06/2026 16:13:43 - INFO - FlagEmbedding.abc.finetune.embedder.AbsRunner -   Training/evaluation parameters AbsEmbedderTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=True,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffer

In [58]:
#Fine-Tuned BGE Recall Metrics
!python /kaggle/input/datasets/madhavkumar244/madhavscuadhn/evaluate_flagembedding.py

Loading model...
Loading weights: 100%|█| 199/199 [00:00<00:00, 1409.38it/s, Materializing param=
✓ Model loaded

Contracts : 102

Evaluating Contracts: 100%|███████████████████| 102/102 [10:05<00:00,  5.93s/it]

FlagEmbedding Evaluation Results

Questions : 1244

Recall@1  : 0.3617
Recall@3  : 0.6342
Recall@5  : 0.7267
Recall@10 : 0.8392
MRR        : 0.5271


In [ ]:
from huggingface_hub import login
login("YOUR_HUGGINGFACE_TOKEN")

In [64]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/kaggle/working/bge_legal",
    repo_id="Madhav2832005/bge-base-legal-retriever",
    repo_type="model"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Madhav2832005/bge-base-legal-retriever/commit/3ccd9dbe9cc5da9d3d0f0395ce2b94ccb3db2585', commit_message='Upload folder using huggingface_hub', commit_description='', oid='3ccd9dbe9cc5da9d3d0f0395ce2b94ccb3db2585', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Madhav2832005/bge-base-legal-retriever', endpoint='https://huggingface.co', repo_type='model', repo_id='Madhav2832005/bge-base-legal-retriever'), pr_revision=None, pr_num=None)